# 01 KEGG pathway selection and gene ID mapping

**Purpose** \
Build, from KEGG only (no expression data), the fixed pathway set and the gene ID mapping used by *both* the GTEx and the TCGA experiments. Everything produced here is saved under `data/kegg/` with the download date, so that the downstream pipeline never queries the network again.

**Outputs**
- `data/kegg/kegg_pathways.tsv`: one row per human KEGG pathway: id, name, BRITE category/subcategory, size, and whether it passes the structural criterion.
- `data/kegg/kegg_pathway_genes.tsv`: pathway → Entrez gene ID pairs.
- `data/kegg/gene_map.tsv`: Entrez → Ensembl (unversioned) → symbol, frozen from mygene.info.
- `data/kegg/README.txt`: download dates and sources.

**Selection criteria**
1. *Structural (KEGG BRITE hierarchy, no data involved):* exclude the "Global and overview maps" (hsa011xx / hsa012xx), the "Human Diseases" category and "Drug Development". These are composite maps that re-assemble pieces of mechanistic pathways and inflate gene sharing; what remains are the mechanistic categories (Metabolism, Genetic / Environmental Information Processing, Cellular Processes, Organismal Systems).
2. *Size:* pathways with 10–150 mapped genes (applied for real in notebook 02, **after** intersecting with the genes present in the expression matrix; here we only preview the counts).

Set `REBUILD = True` to re-download everything; with `False` the frozen files are read from disk.

In [5]:
import os, re, io, json, itertools, collections, datetime
import numpy as np
import pandas as pd
import requests

DATA_DIR = "data/kegg"
os.makedirs(DATA_DIR, exist_ok=True)
REBUILD = not os.path.exists(os.path.join(DATA_DIR, "gene_map.tsv"))
TODAY = datetime.date.today().isoformat()
print("REBUILD =", REBUILD, "| date:", TODAY)

REBUILD = False | date: 2026-09-18


## 1. Download the raw KEGG tables

Four REST endpoints: the pathway list (names), the gene–pathway links, the BRITE hierarchy `hsa00001` (category of each pathway) and the gene list (symbols, used later as a cross-check of the mapping). Raw responses are stored verbatim so the parsing can be re-run offline.

In [6]:
KEGG = "https://rest.kegg.jp"
raw_files = {
    "pathway_list.tsv": f"{KEGG}/list/pathway/hsa",
    "gene_pathway_links.tsv": f"{KEGG}/link/hsa/pathway",
    "brite_hsa00001.txt": f"{KEGG}/get/br:hsa00001",
    "gene_list.tsv": f"{KEGG}/list/hsa",
}
if REBUILD:
    for fname, url in raw_files.items():
        r = requests.get(url, timeout=120); r.raise_for_status()
        with open(os.path.join(DATA_DIR, fname), "w") as f:
            f.write(r.text)
        print(f"downloaded {fname:28s} {len(r.text)/1e6:.1f} MB")
    with open(os.path.join(DATA_DIR, "README.txt"), "w") as f:
        f.write(f"KEGG REST downloaded on {TODAY} from {KEGG}\n")
else:
    print(open(os.path.join(DATA_DIR, "README.txt")).read())

KEGG REST downloaded on 2026-09-18 from https://rest.kegg.jp
gene_map.tsv: mygene.info queried on 2026-09-18 (scopes=entrezgene, fields=ensembl.gene,symbol, species=human)



## 2. Parse pathways, gene links and BRITE categories

The BRITE file is an indented hierarchy: lines starting with `A` are top categories, `B` subcategories, `C` pathways (with the `[PATH:hsaXXXXX]` tag). The global/overview maps (hsa011xx–013xx) are not C-nodes of the hierarchy and are assigned explicitly by prefix. Any other pathway missing from the hierarchy would be flagged `uncategorized` and excluded by criterion 1.

In [15]:
names = pd.read_csv(os.path.join(DATA_DIR, "pathway_list.tsv"), sep="\t", header=None, names=["pid", "name"])
names["name"] = names["name"].str.replace(" - Homo sapiens (human)", "", regex=False)

links = pd.read_csv(os.path.join(DATA_DIR, "gene_pathway_links.tsv"), sep="\t", header=None, names=["pid", "gene"])
links["pid"] = links["pid"].str.replace("path:", "", regex=False)
links["entrez"] = links["gene"].str.replace("hsa:", "", regex=False).astype(int)

cat, sub = {}, {}
A = B = None
for line in open(os.path.join(DATA_DIR, "brite_hsa00001.txt")):
    if line.startswith("A"):
        A = re.sub(r"^\d+ ", "", line[1:].strip())
    elif line.startswith("B"):
        B = re.sub(r"^\d+ ", "", line[1:].strip())
    elif line.startswith("C"):
        m = re.search(r"\[PATH:(hsa\d+)\]", line)
        if m:
            cat[m.group(1)], sub[m.group(1)] = A, B

pw = names.copy()
pw["category"] = pw["pid"].map(cat).fillna("uncategorized")
pw["subcategory"] = pw["pid"].map(sub).fillna("uncategorized")
# the global/overview maps (hsa011xx-013xx) are not listed as C-nodes in BRITE: assign them explicitly
ov = pw["pid"].str.match(r"hsa01[1-3]")
pw.loc[ov, "category"], pw.loc[ov, "subcategory"] = "Metabolism", "Global and overview maps"
pw["n_genes"] = pw["pid"].map(links.groupby("pid").size()).fillna(0).astype(int)
pw = pw[pw["n_genes"] > 0].reset_index(drop=True)

print(f"{len(pw)} human pathways with genes, {links['entrez'].nunique()} distinct Entrez genes, {len(links)} gene-pathway pairs")
pw["category"].value_counts()

372 human pathways with genes, 9423 distinct Entrez genes, 39577 gene-pathway pairs


category
Human Diseases                          96
Metabolism                              95
Organismal Systems                      86
Environmental Information Processing    37
Genetic Information Processing          33
Cellular Processes                      25
Name: count, dtype: int64

## 3. Criterion 1: structural filter

Excluded: 
- the **Global and overview maps** subcategory (hsa011xx / hsa012xx, e.g. "Metabolic pathways" with approx. 1,500 genes)
- the whole **Human Diseases** category (hsa05xxx: disease maps are assemblies of signalling modules, e.g. "Pathways in cancer" with 500+ genes) and **Drug Development** 
- The `uncategorized` pathways are excluded too.

In [16]:
excluded_cats = {"Human Diseases", "Drug Development", "uncategorized"}
pw["pass_criterion1"] = (~pw["category"].isin(excluded_cats)) & (pw["subcategory"] != "Global and overview maps")

print("Dropped by criterion 1:")
print(pw.loc[~pw["pass_criterion1"]].groupby(["category", "subcategory"]).size().to_string())
kept1 = pw[pw["pass_criterion1"]]
print(f"\nKept: {len(kept1)} pathways in {kept1['category'].nunique()} categories")
print(kept1.groupby("category")["n_genes"].describe()[["count", "min", "50%", "max"]].astype(int))

Dropped by criterion 1:
category        subcategory                    
Human Diseases  Cancer: overview                   11
                Cancer: specific types             17
                Cardiovascular disease              7
                Drug resistance: antineoplastic     4
                Endocrine and metabolic disease     8
                Immune disease                      8
                Infectious disease: bacterial      11
                Infectious disease: parasitic       6
                Infectious disease: viral          12
                Neurodegenerative disease           7
                Substance dependence                5
Metabolism      Global and overview maps            9

Kept: 267 pathways in 5 categories
                                      count  min  50%  max
category                                                  
Cellular Processes                       25   32  157  260
Environmental Information Processing     37   29  154  404
Genetic 

## 4. Gene ID mapping (Entrez → Ensembl), frozen

KEGG uses NCBI Entrez IDs (`hsa:7157`), the Xena expression matrix uses Ensembl gene IDs from GENCODE v23 (`ENSG00000141510.11`). We query mygene.info **once**, for the complete list of Entrez IDs that appear in any KEGG pathway, and save the answer. The frozen table is the reference for the whole project; the query date is recorded in `README.txt`.

Rules for ambiguous cases (they matter for reproducibility, so they are explicit here):
- Ensembl versions are dropped (`ENSG...` only); notebook 02 does the same on the Xena row names before joining.
- An Entrez ID with several Ensembl IDs keeps **all** of them here; notebook 02 keeps the one with the highest mean expression among those present in the matrix (usually only one is present).
- Entrez IDs with no Ensembl ID (e.g. pseudogenes, withdrawn records) are dropped: they cannot be found in the matrix anyway.

In [17]:
all_entrez = sorted(links["entrez"].unique())
map_path = os.path.join(DATA_DIR, "gene_map.tsv")
if REBUILD:
    import mygene
    mg = mygene.MyGeneInfo()
    res = mg.querymany([str(e) for e in all_entrez], scopes="entrezgene", species="human",
                       fields="ensembl.gene,symbol", as_dataframe=False, verbose=False)
    rows = []
    for r in res:
        if r.get("notfound") or "ensembl" not in r:
            continue
        ens = r["ensembl"]
        ens = ens if isinstance(ens, list) else [ens]
        for e in ens:
            rows.append((int(r["query"]), e["gene"], r.get("symbol", "")))
    gene_map = pd.DataFrame(rows, columns=["entrez", "ensembl", "symbol"]).drop_duplicates()
    gene_map.to_csv(map_path, sep="\t", index=False)
    with open(os.path.join(DATA_DIR, "README.txt"), "a") as f:
        f.write(f"gene_map.tsv: mygene.info queried on {TODAY} (scopes=entrezgene, fields=ensembl.gene,symbol, species=human)\n")
else:
    gene_map = pd.read_csv(map_path, sep="\t")

n_mapped = gene_map["entrez"].nunique()
multi = gene_map.groupby("entrez").size()
print(f"Entrez IDs in KEGG pathways: {len(all_entrez)}")
print(f"  with >=1 Ensembl ID:      {n_mapped} ({100*n_mapped/len(all_entrez):.1f}%)")
print(f"  with >1 Ensembl ID:       {(multi > 1).sum()}  (max {multi.max()})")
print(f"  unmapped:                 {len(all_entrez) - n_mapped}")
gene_map.head()

Entrez IDs in KEGG pathways: 9423
  with >=1 Ensembl ID:      8883 (94.3%)
  with >1 Ensembl ID:       736  (max 42)
  unmapped:                 540


,entrez,ensembl,symbol
0,2,ENSG00000175899,A2M
1,9,ENSG00000171428,NAT1
2,10,ENSG00000156006,NAT2
3,15,ENSG00000129673,AANAT
4,16,ENSG00000090861,AARS1


**Cross-check with KEGG's own symbols.** `list/hsa` gives, for each Entrez ID, KEGG's symbol list. A mismatch between the mygene symbol and every KEGG symbol flags a possibly wrong record (typically renamed genes; they are still correct mappings, but worth a look).

In [18]:
kegg_genes = pd.read_csv(os.path.join(DATA_DIR, "gene_list.tsv"), sep="\t", header=None,
                         names=["gene", "type", "position", "description"], quoting=3)
kegg_genes["entrez"] = kegg_genes["gene"].str.replace("hsa:", "", regex=False).astype(int)
kegg_genes["symbols"] = kegg_genes["description"].str.split(";").str[0].str.split(", ")
sym = kegg_genes.set_index("entrez")["symbols"]
chk = gene_map.assign(kegg_symbols=gene_map["entrez"].map(sym))
chk = chk[chk["kegg_symbols"].notna()]
chk["match"] = [s in ks for s, ks in zip(chk["symbol"], chk["kegg_symbols"])]
print(f"symbol agreement mygene vs KEGG: {chk['match'].mean()*100:.1f}% of {len(chk)} rows")
chk.loc[~chk["match"], ["entrez", "ensembl", "symbol", "kegg_symbols"]].head(10)

symbol agreement mygene vs KEGG: 98.9% of 10581 rows


,entrez,ensembl,symbol,kegg_symbols
2959,4691,ENSG00000115053,NUCLEOLIN,"[NCL, C23, Nsr1]"
5064,8808,ENSG00000115598,IL36R,"[IL1RL2, IL-1Rrp2, IL-36R, IL1R-rp2, IL1RRP2]"
9649,339983,ENSG00000185818,ASPNAT,"[NAT8L, CML3, NACED, NAT8-LIKE]"
10405,100653049,ENSG00000262748,LOC100653049,"[keratin, type I cuticular Ha4 isoform X1]"
10429,102723407,ENSG00000275063,LOC102723407,[immunoglobulin heavy variable 4-38-2-like]
10430,102723475,ENSG00000276289,LOC102723475,[KCNE1B]
10431,102723532,ENSG00000274792,LOC102723532,[olfactory receptor 4N4]
10434,102723573,ENSG00000288164,LOC102723573,[40S ribosomal protein S8-like]
10437,102723996,ENSG00000277117,LOC102723996,[ICOS ligand precursor]
10438,102724159,ENSG00000275464,LOC102724159,[periodic tryptophan protein 2 homolog]


## 5. Preview of criterion 2 (size) on the mapped genes

Sizes are recomputed on mapped genes only (an unmapped gene can never reach the model). The definitive table is produced in notebook 02 after the expression filter; this preview is what we use to choose the starting thresholds.

In [19]:
mapped_entrez = set(gene_map["entrez"])
links_m = links[links["entrez"].isin(mapped_entrez) & links["pid"].isin(kept1["pid"])]
genes_by_pw = links_m.groupby("pid")["entrez"].apply(set)
size_m = genes_by_pw.apply(len)

rows = []
for lo, hi in [(10, 100), (10, 150), (10, 200), (10, 250), (15, 150)]:
    sel = size_m[(size_m >= lo) & (size_m <= hi)].index
    cnt = collections.Counter(itertools.chain.from_iterable(genes_by_pw[p] for p in sel))
    rows.append({"range": f"[{lo}, {hi}]", "pathways": len(sel), "genes": len(cnt),
                 "n_max": int(size_m[sel].max()), "pathways_per_gene": round(np.mean(list(cnt.values())), 2),
                 "local_attention_cost (K*n_max^2, M)": round(len(sel) * size_m[sel].max() ** 2 / 1e6, 1)})
pd.DataFrame(rows)

,range,pathways,genes,n_max,pathways_per_gene,"local_attention_cost (K*n_max^2, M)"
0,"[10, 100]",160,4226,100,1.90,1.6
1,"[10, 150]",207,5730,149,2.40,4.6
2,"[10, 200]",229,6628,199,2.64,9.1
3,"[10, 250]",242,7537,250,2.71,15.1
4,"[15, 150]",202,5698,149,2.41,4.5


In [ ]:
# Pathways lost with the upper bound at 150. To be stated explicitly and discussed
lost = kept1.set_index("pid").loc[size_m[size_m > 150].index].assign(n_mapped=size_m[size_m > 150])
lost.sort_values("n_mapped", ascending=False)[["name", "subcategory", "n_mapped"]]

,name,subcategory,n_mapped
pid,,,
hsa04740,Olfactory transduction,Sensory system,448
hsa04519,Cadherin signaling,Signaling molecules and interaction,404
hsa04080,Neuroactive ligand-receptor interaction,Signaling molecules and interaction,371
hsa04151,PI3K-Akt signaling pathway,Signal transduction,362
hsa04010,MAPK signaling pathway,Signal transduction,301
hsa04517,IgSF CAM signaling,Signaling molecules and interaction,299
hsa04060,Cytokine-cytokine receptor interaction,Signaling molecules and interaction,296
hsa04145,Phagocytosis,Transport and catabolism,258
hsa04020,Calcium signaling pathway,Signal transduction,254


## 6. Save the frozen tables

In [22]:
pw.to_csv(os.path.join(DATA_DIR, "kegg_pathways.tsv"), sep="\t", index=False)
links[["pid", "entrez"]].to_csv(os.path.join(DATA_DIR, "kegg_pathway_genes.tsv"), sep="\t", index=False)
print("saved:", sorted(os.listdir(DATA_DIR)))
print(open(os.path.join(DATA_DIR, "README.txt")).read())

saved: ['README.txt', 'brite_hsa00001.txt', 'gene_list.tsv', 'gene_map.tsv', 'gene_pathway_links.tsv', 'kegg_pathway_genes.tsv', 'kegg_pathways.tsv', 'pathway_list.tsv']
KEGG REST downloaded on 2026-09-18 from https://rest.kegg.jp
gene_map.tsv: mygene.info queried on 2026-09-18 (scopes=entrezgene, fields=ensembl.gene,symbol, species=human)

